# Initialize

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import numpy as np
import random
import plotly.express as px

In [2]:
# Load JSON file (update the path if needed)
with open('CSE23_schd.json') as f:
    data = json.load(f)

conference_year = "2023"

In [3]:
def linear_to_srgb(c):
    # Piecewise sRGB EOTF
    a = 0.055
    threshold = 0.0031308
    srgb = np.where(c <= threshold, 12.92 * c, (1 + a) * np.power(np.clip(c, 0, None), 1/2.4) - a)
    return np.clip(srgb, 0, 1)

In [4]:
import colorsys

def prepareTimeline(data, map, modelName = "", fileName = "schedule.html"):
  # Flatten into a DataFrame
  events = list(data.values())

  # Keep only those with codes starting with 'MS'
  events = [e for e in events if e.get("code", "").startswith("MS")]

  # Clean missing data
  for e in events:
    if map.get(e["code"]) is not None:
      e["clusterID"] = int(map[e["code"]])
    else:
      e["clusterID"] = -1
    if(pd.isna(e["room"])):
      e["room"] = "NA"

  df = pd.DataFrame(events) # Use the filtered and cleaned events list


  # Filter out rows where clusterID is -1 before converting to int
  df = df[df['clusterID'] != -1].copy()


  # 2) Parse datetimes (2025 as the year)
  df['start'] = pd.to_datetime(df['day'] + ' ' + df['begin_time'] + f" {conference_year}",
                              format='%A, %B %d %I:%M %p %Y')
  df['end']   = pd.to_datetime(df['day'] + ' ' + df['end_time']   + f" {conference_year}",
                              format='%A, %B %d %I:%M %p %Y')

  # 3) Convert clusterID to string so it’s categorical
  df['clusterID'] = df['clusterID'].astype(int).astype(str)

  # Sort by clusterID to potentially influence legend order
  df = df.sort_values(by='clusterID')


  # 4) Build a discrete color map using HSV
  cluster_vals = sorted([int(c) for c in df['clusterID'].unique()]) # Get unique cluster IDs as integers
  num_clusters = len(cluster_vals)
  color_map = {}
  for i, cid in enumerate(cluster_vals):
      # Map cluster ID to hue (0-1), keeping saturation and value constant
      hue = i / num_clusters
      rgb_color = colorsys.hsv_to_rgb(hue, 0.8, 0.8) # Using saturation and value of 0.8
      # Convert RGB to hex color
      hex_color = '#%02x%02x%02x' % (int(rgb_color[0]*255), int(rgb_color[1]*255), int(rgb_color[2]*255))
      color_map[str(cid)] = hex_color # Store with string cluster ID

  # 5) Create the timeline with discrete colors
  fig = px.timeline(
      df,
      x_start="start",
      x_end="end",
      y="room",
      color="clusterID",
      color_discrete_map=color_map, # Use the precalculated discrete color map
      # text="code",
      category_orders={"room": sorted(df['room'].unique())}, # Keep room order sorted
      title=f"Conference Schedule {modelName}"
  )
  fig.update_yaxes(autorange="reversed")

  # 6) (Optional) add day separators

  # Set the x-axis range to start at the beginning of the first day and format ticks
  min_date = df['start'].min().normalize()
  max_date = df['end'].max().normalize()
  fig.update_layout(
      xaxis=dict(
          range=[min_date, max_date],
          tickformat='%b %d',  # Format ticks to show month and day
          dtick='d'            # Place ticks at the start of each day
      ),
      legend=dict(
          traceorder="normal" # This might help maintain the order from category_orders
      )
  )

  shapes = []
  for day in sorted(df['start'].dt.normalize().unique()):
      sep = day + pd.Timedelta(days=1)
      shapes.append({
          "type": "line",
          "x0": sep, "x1": sep,
          "y0": -0.5, "y1": len(df['room'].unique()) - 0.5,
          "line": {"color": "gray", "width": 1, "dash": "dash"}
      })
  fig.update_layout(shapes=shapes)

  # 7) Export to standalone HTML
  fig.write_html(fileName, full_html=True)
  print(f"→ {fileName} written with discrete cluster colors.")

## Load from file (Map MS: CitationCount)

In [5]:
map_cluster_all_title = "SBERT - PCA 10 - Normalized - KMeans(30)"
map_cluster_all = {'MS68': 0, 'MS71': 0, 'MS101': 0, 'MS106': 0, 'MS112': 0, 'MS136': 0, 'MS142': 0, 'MS176': 0, 'MS193': 0, 'MS201': 0, 'MS228': 0, 'MS283': 0, 'MS291': 0, 'MS352': 0, 'MS380': 0, 'MS387': 0, 'MS404': 0, 'MS17': 1, 'MS29': 1, 'MS72': 1, 'MS87': 1, 'MS127': 1, 'MS163': 1, 'MS195': 1, 'MS220': 1, 'MS227': 1, 'MS313': 1, 'MS345': 1, 'MS1': 2, 'MS5': 2, 'MS44': 2, 'MS77': 2, 'MS113': 2, 'MS134': 2, 'MS149': 2, 'MS189': 2, 'MS217': 2, 'MS225': 2, 'MS246': 2, 'MS8': 2, 'MS254': 2, 'MS275': 2, 'MS282': 2, 'MS301': 2, 'MS321': 2, 'MS333': 2, 'MS407': 2, 'MS11': 3, 'MS15': 3, 'MS31': 3, 'MS37': 3, 'MS54': 3, 'MS66': 3, 'MS80': 3, 'MS105': 3, 'MS122': 3, 'MS145': 3, 'MS156': 3, 'MS159': 3, 'MS180': 3, 'MS182': 3, 'MS216': 3, 'MS231': 3, 'MS234': 3, 'MS259': 3, 'MS272': 3, 'MS309': 3, 'MS323': 3, 'MS376': 3, 'MS398': 3, 'MS399': 3, 'MS411': 3, 'MS34': 4, 'MS46': 4, 'MS53': 4, 'MS76': 4, 'MS82': 4, 'MS85': 4, 'MS91': 4, 'MS120': 4, 'MS151': 4, 'MS160': 4, 'MS185': 4, 'MS260': 4, 'MS361': 4, 'MS406': 4, 'MS51': 5, 'MS88': 5, 'MS124': 5, 'MS150': 5, 'MS184': 5, 'MS191': 5, 'MS244': 5, 'MS256': 5, 'MS258': 5, 'MS279': 5, 'MS299': 5, 'MS304': 5, 'MS322': 5, 'MS337': 5, 'MS365': 5, 'MS392': 5, 'MS400': 5, 'MS26': 6, 'MS35': 6, 'MS98': 6, 'MS99': 6, 'MS104': 6, 'MS132': 6, 'MS139': 6, 'MS167': 6, 'MS170': 6, 'MS177': 6, 'MS203': 6, 'MS262': 6, 'MS263': 6, 'MS280': 6, 'MS303': 6, 'MS308': 6, 'MS324': 6, 'MS341': 6, 'MS368': 6, 'MS20': 7, 'MS28': 7, 'MS52': 7, 'MS63': 7, 'MS64': 7, 'MS93': 7, 'MS94': 7, 'MS125': 7, 'MS148': 7, 'MS172': 7, 'MS179': 7, 'MS207': 7, 'MS264': 7, 'MS335': 7, 'MS372': 7, 'MS397': 7, 'MS418': 7, 'MS24': 8, 'MS61': 8, 'MS78': 8, 'MS115': 8, 'MS158': 8, 'MS190': 8, 'MS326': 8, 'MS336': 8, 'MS384': 8, 'MS420': 8, 'MS10': 9, 'MS45': 9, 'MS173': 9, 'MS450': 9, 'MS314': 9, 'MS13': 10, 'MS169': 10, 'MS202': 10, 'MS245': 10, 'MS281': 10, 'MS285': 10, 'MS385': 10, 'MS3': 11, 'MS16': 11, 'MS103': 11, 'MS138': 11, 'MS215': 11, 'MS247': 11, 'MS252': 11, 'MS310': 11, 'MS343': 11, 'MS353': 11, 'MS388': 11, 'MS390': 11, 'MS7': 12, 'MS40': 12, 'MS62': 12, 'MS135': 12, 'MS162': 12, 'MS168': 12, 'MS175': 12, 'MS200': 12, 'MS211': 12, 'MS224': 12, 'MS329': 12, 'MS375': 12, 'MS378': 12, 'MS382': 12, 'MS405': 12, 'MS410': 12, 'MS413': 12, 'MS27': 13, 'MS33': 13, 'MS69': 13, 'MS90': 13, 'MS96': 13, 'MS108': 13, 'MS130': 13, 'MS265': 13, 'MS292': 13, 'MS312': 13, 'MS316': 13, 'MS325': 13, 'MS339': 13, 'MS347': 13, 'MS350': 13, 'MS367': 13, 'MS371': 13, 'MS379': 13, 'MS403': 13, 'MS14': 14, 'MS50': 14, 'MS102': 14, 'MS137': 14, 'MS164': 14, 'MS196': 14, 'MS218': 14, 'MS238': 14, 'MS250': 14, 'MS274': 14, 'MS289': 14, 'MS306': 14, 'MS338': 14, 'MS366': 14, 'MS374': 14, 'MS32': 15, 'MS84': 15, 'MS373': 15, 'MS70': 15, 'MS121': 15, 'MS9': 15, 'MS41': 15, 'MS110': 15, 'MS133': 15, 'MS192': 15, 'MS199': 15, 'MS213': 15, 'MS243': 15, 'MS248': 15, 'MS257': 15, 'MS278': 15, 'MS293': 15, 'MS294': 15, 'MS307': 15, 'MS340': 15, 'MS65': 16, 'MS95': 16, 'MS126': 16, 'MS129': 16, 'MS152': 16, 'MS157': 16, 'MS161': 16, 'MS183': 16, 'MS194': 16, 'MS206': 16, 'MS214': 16, 'MS222': 16, 'MS236': 16, 'MS249': 16, 'MS286': 16, 'MS370': 16, 'MS23': 17, 'MS36': 17, 'MS48': 17, 'MS57': 17, 'MS60': 17, 'MS75': 17, 'MS111': 17, 'MS116': 17, 'MS295': 17, 'MS298': 17, 'MS358': 17, 'MS360': 17, 'MS393': 17, 'MS396': 17, 'MS401': 17, 'MS4': 18, 'MS19': 18, 'MS21': 18, 'MS43': 18, 'MS55': 18, 'MS58': 18, 'MS147': 18, 'MS174': 18, 'MS197': 18, 'MS226': 18, 'MS240': 18, 'MS297': 18, 'MS330': 18, 'MS415': 18, 'MS83': 19, 'MS140': 19, 'MS146': 19, 'MS154': 19, 'MS268': 19, 'MS327': 19, 'MS348': 19, 'MS386': 19, 'MS6': 20, 'MS38': 20, 'MS39': 20, 'MS81': 20, 'MS86': 20, 'MS117': 20, 'MS141': 20, 'MS208': 20, 'MS241': 20, 'MS242': 20, 'MS251': 20, 'MS261': 20, 'MS269': 20, 'MS300': 20, 'MS332': 20, 'MS362': 20, 'MS364': 20, 'MS419': 20, 'MS30': 21, 'MS97': 21, 'MS107': 21, 'MS131': 21, 'MS153': 21, 'MS155': 21, 'MS186': 21, 'MS187': 21, 'MS237': 21, 'MS273': 21, 'MS315': 21, 'MS349': 21, 'MS377': 21, 'MS383': 21, 'MS395': 21, 'MS412': 21, 'MS421': 21, 'MS100': 22, 'MS123': 22, 'MS166': 22, 'MS171': 22, 'MS204': 22, 'MS235': 22, 'MS276': 22, 'MS363': 22, 'MS394': 22, 'MS230': 23, 'MS255': 23, 'MS288': 23, 'MS302': 23, 'MS320': 23, 'MS356': 23, 'MS391': 23, 'MS409': 23, 'MS25': 24, 'MS67': 24, 'MS114': 24, 'MS284': 24, 'MS311': 24, 'MS344': 24, 'MS354': 24, 'MS389': 24, 'MS47': 25, 'MS56': 25, 'MS74': 25, 'MS143': 25, 'MS178': 25, 'MS188': 25, 'MS223': 25, 'MS253': 25, 'MS290': 25, 'MS355': 25, 'MS417': 25, 'MS92': 26, 'MS144': 26, 'MS165': 26, 'MS198': 26, 'MS210': 26, 'MS219': 26, 'MS229': 26, 'MS232': 26, 'MS266': 26, 'MS267': 26, 'MS270': 26, 'MS296': 26, 'MS305': 26, 'MS334': 26, 'MS342': 26, 'MS357': 26, 'MS381': 26, 'MS416': 26, 'MS18': 27, 'MS22': 27, 'MS59': 27, 'MS128': 27, 'MS181': 27, 'MS205': 27, 'MS221': 27, 'MS233': 27, 'MS277': 27, 'MS287': 27, 'MS359': 27, 'MS369': 27, 'MS402': 27, 'MS2': 28, 'MS12': 28, 'MS42': 28, 'MS79': 28, 'MS89': 28, 'MS118': 28, 'MS209': 28, 'MS212': 28, 'MS328': 28, 'MS422': 28, 'MS73': 29, 'MS109': 29, 'MS119': 29, 'MS271': 29, 'MS317': 29, 'MS351': 29, 'MS408': 29, 'MS414': 29}

In [6]:
with open("map_ms_cluster_all.json", "w") as f:
    json.dump(map_cluster_all, f)

In [7]:
MAP_FILE = "CSE23_B1_output_ms_map.json"
# MAP_FILE = "map_ms_cluster_all.json"

with(open(MAP_FILE, "r")) as f:
    map_cluster_all = json.load(f)

map_cluster_all_title = "CSE23 - B1 - Clustering (SBERT + PCA + KMeans)"
MODEL_NAME = "CSE_2023_MS_ClusterMap"

FileNotFoundError: [Errno 2] No such file or directory: 'CSE23_B1_output_ms_map.json'

## Clustering Visualization

In [ ]:
prepareTimeline(data, map_cluster_all, modelName = map_cluster_all_title, fileName = "CSE23_schedule_B1_clusters.html")